In [ ]:
import os
from dotenv import load_dotenv
from getpass import getpass
from openai import OpenAI

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
grok_api_key = os.getenv("GROK_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

if not openai_api_key:
    print("No API Key for OpenAI was found. Please recheck")
    openai_api_key = getpass("Enter OpenAI API Key: ")

if not grok_api_key:
    print("No API Key for Grok was found. Please recheck")
    groq_api_key = getpass("Enter Grok API Key: ")

if not deepseek_api_key:
    print("No API Key for Deepseek was found. Please recheck")
    groq_api_key = getpass("Enter Deepseek API Key: ")

if not anthropic_api_key:
    print("No API Key for Groq was found. Please recheck")
    groq_api_key = getpass("Enter Anthropic API Key: ")

In [ ]:
deepseek_base_url = "https://api.deepseek.com"
grok_base_url = "https://api.x.ai/v1"
anthropic_base_url = "https://api.anthropic.com/v1"

deepseek_model = "deepseek-flash"
grok_model = "grok-4.3"
anthropic_model = "claude-haiku-4-5-20251001"

deepseek_agent = OpenAI(base_url=deepseek_base_url, api_key=deepseek_api_key)
grok_agent = OpenAI(base_url=grok_base_url, api_key=grok_api_key)
anthropic_agent = OpenAI(base_url=anthropic_base_url, api_key=anthropic_api_key)

agents = [(deepseek_agent, deepseek_model), (grok_agent, grok_model), (anthropic_agent, anthropic_model)]

In [ ]:
from random import choice

# define roles for each agent
roles = {role: None for role in ["judge", "pro", "anti"]}
idx = list(range(0,3))

# randomly assign a judge
judge_idx = choice(idx)
idx.remove(judge_idx)

# randomly assign debaters
pro_idx = choice(idx)
idx.remove(pro_idx)
anti_idx = choice(idx) if len(idx) > 1 else idx[0]

# assign finalized roles to agents
roles["judge"] = agents[judge_idx]
roles["pro"] = agents[pro_idx]
roles["anti"] = agents[anti_idx]

print(f"The judge for this debate session will be {roles["judge"][1].split("-")[0].title()}.")
print(f"Agent {roles["pro"][1].split("-")[0].title()} will debate for the topic.")
print(f"Agent {roles["anti"][1].split("-")[0].title()} will debate against it.")
print("Let us ask some of the audience to pick a random topic worth debating...")
print("OpenAI! Found you! You are chosen to pick a topic for today's debate! Don't be shy. Give us a sentence!")


In [ ]:
openai_agent = OpenAI()
openai_system = """
You are a random audience in a debate show where audiences choose which topic to debate.
You are curious and can have a variety of interests.
You are free to choose what kind of person you are.
You are only here for one thing: satisfy your curiosity and watch the debate for entertainment.
"""
openai_messages = [{"role": "system", "content": openai_system}]
openai_user = """
You will be watching a debate between other LLM agents.
You are given the priviledge to think of a topic worth debating.
Make it simple and comprehensible. It must be a value-laden single sentence.

Here are some examples:
'Pineapple on pizza is a good idea.'
'Social Darwinism is totally baseless.'
'The Vietnam War was unjustifiable.'

They can be general topics, a trivial subject or any currents events before 2024. Controversy is welcome.

You must only answer in a single sentence. Please refer to the aforementioned examples.
"""
openai_messages.append({"role": "user", "content": openai_user})


response = openai_agent.chat.completions.create(
    model="gpt-4.1-mini",
    messages=openai_messages
)

debate_topic = response.choices[0].message.content
print(f"OpenAI: {debate_topic}")
print(f"And we found a topic: {debate_topic}")
print(f"You are given 5 rounds to debate the topic.")
print(f"Debaters {roles["pro"][1].split("-")[0].title()} and {roles["anti"][1].split("-")[0].title()} can only argue in one sentence.")
print(f"At the end of the show, the judge {roles["judge"][1].split("-")[0].title()} will announce the winner.")

In [ ]:
debate_session = []
ROUNDS = 5

pro_debater_system = f"""
You are chosen to debate for this topic: {debate_topic}. You will argue for this topic.
You are already well researched you don't need to look up for supporting arguments.
"""

anti_debater_system = f"""
You are chosen to debate for this topic: {debate_topic}. You will argue against this topic.
You will also look for any loopholes in arguments provided by your opponent.
You don't need to research to come up with an answer. You rely on implicit reasoning for counterarguments.
"""

judge_system = f"""
You are chosen to judge for tonight's debate show. The topic will be '{debate_topic[:-1]}'.
At the end of the show, you will merit each argument for and against tonight's debate topic.
You will determine the more reasonable debater.
You are already a well-researched and well-read person.
You will use your own well-formed judgments and sharp logic to determine which side presented his/her case more objectively.
"""

def format_arguments(arguments):
    formatted = ""
    for i, argument in enumerate(arguments):
        formatted += "\n"
        formatted += "Against:" if i % 2 else "Pro:"
        formatted += f" {argument}"
    return formatted

In [ ]:
def argue_for(pro=True):
    messages = [{"role": "system", "content": pro_debater_system if pro else anti_debater_system}]
    callback_user = f"""
    You are debating for this topic: {debate_topic}. This is how the debate session is ongoing:
    {format_arguments(debate_session)}
    \nBased on the latest argument {"against" if pro else "supporting"} the topic, how will you respond?
    Remember to avoid any personal attacks or unnecessary soliloquy.
    Limit your argument up to four sentences. Make it make sense to everyone watching tonight.

    ## Format
    <Sentence 1>. <Sentence 2>. <Sentence 3>. <Sentence 4>.

    Please respond as presented by desired format.
    Don't reply in markdowns like presented. Think of `<Sentence 1>` as an actual sentence.

    Follow the format strictly.
    """
    messages.append({"role": "user", "content": callback_user})
    callback_model = roles["pro"][1] if pro else roles["anti"][1]
    callback_agent = roles["pro"][0] if pro else roles["anti"][0]
    
    response = callback_agent.chat.completions.create(
        model=callback_model,
        messages=messages
    )
    argument = response.choices[0].message.content
    print(argument)
    debate_session.append(argument)

print("The debate starts now!")

for i in range(ROUNDS * 2):
    if not i % 2:
        print(f"ROUND {(i // 2) + 1}!")
        print(f"{roles["pro"][1].split("-")[0].title()}?")
        argue_for()
        continue
    print(f"{roles["anti"][1].split("-")[0].title()}?")
    argue_for(False)
    print("=" * 500)

In [ ]:
print(f"{ROUNDS} rounds are up! It's time to judge who won the debate.")

judge_messages = [{"role": "system", "content": judge_system}]
user_judge = f"""
The debate lasted for 5 sessions. Tonight's debate topic is {debate_topic}.
Agent arguing for the topic is {roles["pro"][1].split("-")[0].title()}.
Arguing against the topic is {roles["anti"][1].split("-")[0].title()}.

Here is the entire debate session for tonight's show:
{format_arguments(debate_session)}

Weight each argument presented and determine who is tonight's debate winner.
Do not hesitate to demerit any form of personal attack throughout the session.
Also consider the grammar and spelling as valid points.

In your response, address each debater by their name and in 2 concise sentences,
state your observations of their argument throughout tonight's debate session.

Then as a final verdict, name the winner in one final sentence.
"""

judge_messages.append({"role": "user", "content": user_judge})

judge_agent, judge_model = roles["judge"]
judge_response = judge_agent.chat.completions.create(
    model=judge_model,
    messages=judge_messages
)

final_verdict = judge_response.choices[0].message.content

print("Judge?")
print(final_verdict)
print("Applauses...")
print("I hope you enjoyed tonight's debate show! Stay tuned for the next episode.")